# 5.2 🔐 Configuración Profesional y Gestión de Secretos

En sistemas reales **el código no debe contener secretos**.

En proyectos de analítica o backend es común ver cosas como:

```python
API_KEY = "abc123"
DATABASE_PASSWORD = "mypassword"
```

Esto es peligroso porque:

- Termina en GitHub (exposición pública o interna)
- No se puede rotar fácil (cambiar la key sin tocar código)
- Rompe ambientes (dev/test/prod)
- Aumenta la superficie de ataque

---

## 🎯 Objetivo

Aprender a separar:

- **Código** (lógica estable)
- **Configuración** (valores por ambiente)
- **Secretos** (API keys, passwords, tokens)

y dejar un **cheatsheet** aplicable al proyecto SmartPortfolio.


## 🧭 Arquitectura de Configuración

```{mermaid}
flowchart LR
    A[Codigo Fuente] -->|No secrets| D[Aplicacion]
    B[Variables de Entorno] --> C[Settings Object]
    C --> D
    E[env file - dev only] --> B
```


## 1) Setup tangible: `.venv` con Poetry

### ✅ Crear el entorno virtual (dentro del repo)

En la raíz del proyecto (donde está `pyproject.toml`):

```bash
poetry config virtualenvs.in-project true
poetry install
```

Esto crea la carpeta:

```text
.venv/
```

### ✅ Verificar que estás en el entorno correcto

```bash
poetry env info
poetry run python -V
```

### ✅ (Opcional, recomendado) Kernel de Jupyter usando `.venv`

Si vas a ejecutar notebooks localmente:

```bash
poetry add ipykernel --group dev
poetry run python -m ipykernel install --user --name smartportfolio --display-name "SmartPortfolio (.venv)"
```

Luego seleccionas el kernel **SmartPortfolio (.venv)** en Jupyter.


## 2) Variables de entorno: el estándar profesional

Las variables de entorno te permiten configurar la aplicación sin cambiar el código.

En Linux/Mac:

```bash
export ALPHA_VANTAGE_KEY="tu_key"
export ENVIRONMENT="dev"
poetry run python tu_script.py
```

En Windows PowerShell:

```powershell
$env:ALPHA_VANTAGE_KEY="tu_key"
$env:ENVIRONMENT="dev"
poetry run python tu_script.py
```


In [1]:
import os

api_key = os.getenv("ALPHA_VANTAGE_KEY")
environment = os.getenv("ENVIRONMENT", "dev")

print("Environment:", environment)
print("API Key loaded:", api_key is not None)


Environment: dev
API Key loaded: True


## 3) `.env` (solo para desarrollo) + `python-dotenv`

En desarrollo es normal usar un archivo `.env` para no exportar variables todo el tiempo.

### ✅ Crear `.env` (NO se sube a Git)

En la raíz del repo:

```text
ALPHA_VANTAGE_KEY=tu_key
ENVIRONMENT=dev
```

### ✅ Crear `.env.example` (SÍ se sube a Git)

```text
ALPHA_VANTAGE_KEY=your_key_here
ENVIRONMENT=dev
```

### Instalar `python-dotenv`

```bash
poetry add python-dotenv --group dev
```


In [2]:
from dotenv import load_dotenv
import os

# Loads variables from a local .env file (if present)
load_dotenv()

api_key = os.getenv("ALPHA_VANTAGE_KEY")
environment = os.getenv("ENVIRONMENT", "dev")

print("Environment:", environment)
print("API Key loaded:", api_key is not None)


Environment: dev
API Key loaded: True


## 4) `.gitignore`: cómo evitar subir secretos

En la raíz del repo, crea o edita `.gitignore` y agrega:

```text
# --- Python / env ---
.venv/
__pycache__/
*.pyc

# --- Secrets / config ---
.env
.env.*
!.env.example

# --- Jupyter ---
.ipynb_checkpoints/
```

### 🚨 Si ya subiste `.env` por accidente

1) Déjalo en `.gitignore`  
2) Sácalo del tracking:

```bash
git rm --cached .env
git commit -m "chore: stop tracking .env"
```


## 5) Patrón Settings (simple, sin dependencias)

Centralizar configuración en un objeto evita repetir `os.getenv(...)` por todo el código.

> Este patrón mantiene el dominio limpio: la configuración vive “afuera”.


In [3]:
from dataclasses import dataclass
import os
from dotenv import load_dotenv

# Load local .env if present (dev only)
load_dotenv()


@dataclass(frozen=True, slots=True)
class Settings:
    """
    Application settings loaded from environment variables.

    This object centralizes configuration so the rest of the
    system does not call os.getenv everywhere.
    """

    alpha_vantage_key: str
    environment: str = "dev"

    @classmethod
    def load(cls) -> "Settings":
        """
        Load settings from environment variables and validate them.
        """

        api_key = os.getenv("ALPHA_VANTAGE_KEY")
        env = os.getenv("ENVIRONMENT", "dev")

        if not api_key:
            raise RuntimeError(
                "Missing ALPHA_VANTAGE_KEY. "
                "Define it in environment variables or in a .env file."
            )

        if env not in {"dev", "test", "prod"}:
            raise ValueError(
                f"Invalid ENVIRONMENT '{env}'. Use: dev, test, prod."
            )

        return cls(
            alpha_vantage_key=api_key,
            environment=env,
        )


settings = Settings.load()

print("Environment:", settings.environment)
print("API Key loaded:", settings.alpha_vantage_key[:4] + "****")


Environment: dev
API Key loaded: tu_k****


## 6) Configuración con Pydantic v2: `pydantic-settings`

En Pydantic v2, `BaseSettings` se movió a `pydantic-settings`.

### Instalar

```bash
poetry add pydantic-settings --group dev
```

### Ejemplo (Pydantic v2)

> Este modelo valida configuración y también puede leer `.env` automáticamente.


In [4]:
from __future__ import annotations

try:
    from pydantic_settings import BaseSettings, SettingsConfigDict
except ImportError as exc:
    raise RuntimeError(
        "pydantic-settings is not installed. "
        "Install with: poetry add pydantic-settings --group dev"
    ) from exc


class AppSettings(BaseSettings):
    """Settings validated by Pydantic (v2)."""

    alpha_vantage_key: str
    environment: str = "dev"

    # Reads from .env if present; ignores unknown env vars
    model_config = SettingsConfigDict(env_file=".env", extra="ignore")


app_settings = AppSettings()

print("Environment:", app_settings.environment)
print("API key loaded:", bool(app_settings.alpha_vantage_key))
print("API key prefix:", app_settings.alpha_vantage_key[:4] + "****")

Environment: dev
API key loaded: True
API key prefix: tu_k****


## 7) Uso dentro del sistema: inyección de configuración

La idea es que los componentes que hablan con el mundo externo (providers, DB clients)
reciban configuración por constructor.

```{mermaid}
flowchart LR
    A[Settings] --> B[Provider]
    A --> C[DB Client]
    B --> D[External API]
```


In [5]:
from __future__ import annotations


class AlphaVantageClient:
    """Client responsible for interacting with the Alpha Vantage API."""

    def __init__(self, api_key: str) -> None:
        if not api_key:
            raise ValueError("Alpha Vantage API key is required.")

        self._api_key: str = api_key

    @property
    def masked_key(self) -> str:
        """Return a safe representation of the API key."""
        return self._api_key[:4] + "****"

    def info(self) -> str:
        """Return a diagnostic message about the client configuration."""
        return f"AlphaVantage client configured (key={self.masked_key})"


client = AlphaVantageClient(settings.alpha_vantage_key)

print(client.info())


AlphaVantage client configured (key=tu_k****)


## ✅ Checklist (para el SmartPortfolio Pro)

- [ ] `.env` existe solo en local (no se sube)
- [ ] `.env.example` está en Git como plantilla
- [ ] `.gitignore` bloquea `.env` y `.venv/`
- [ ] La configuración vive en `Settings`
- [ ] Providers reciben settings por inyección (no globales)
- [ ] Keys nunca aparecen en logs completos (solo prefijo)

---

## 🎯 Cierre

En 5.1 protegimos el dominio (inputs coherentes).  
En 5.2 protegimos la infraestructura (secretos y ambientes).

En 5.3 vamos a acelerar descargas masivas (async) para soportar analítica multi‑ticker:
retornos, volatilidad, Bollinger, y Markowitz.
